# 07 — Abundance Calculations from G395M Spectrum

**Purpose:** Test the `jwspecabund` package on a real G395M grating spectrum
with a strong [OIII] 4363 detection, enabling both the direct T_e method
and the Sanders+25 strong-line calibration. Compare results with published values.

**Data:** `stark-rxcj2248-v4_g395m-f290lp_2478_3.spec.fits` — a lensed galaxy
from the RXCJ2248 field at z = 6.1052.

**Date:** 2026-02-19

In [ ]:
import jwspecfit
import jwspecabund
import matplotlib.pyplot as plt
import numpy as np

print(f"jwspecfit   v{jwspecfit.__version__}")
print(f"jwspecabund v{jwspecabund.__version__}")

## 1. Load the G395M spectrum

In [ ]:
spec = jwspecfit.read_fits(
    "../../data/stark-rxcj2248-v4_g395m-f290lp_2478_3.spec.fits",
    z=6.1052,
)

print(f"Grating:    {spec.grating}")
print(f"Pixels:     {spec.n_pix}")
print(f"Wave range: {spec.wave_um.min():.3f} \u2013 {spec.wave_um.max():.3f} \u00b5m")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))
valid = spec.mask_valid()
ax.step(spec.wave_um[valid], spec.flux_ujy[valid], where="mid", lw=0.6, color="0.3")
ax.fill_between(
    spec.wave_um[valid],
    (spec.flux_ujy - spec.err_ujy)[valid],
    (spec.flux_ujy + spec.err_ujy)[valid],
    step="mid", alpha=0.15, color="0.5",
)
ax.set_xlabel(r"Wavelength [$\mu$m]")
ax.set_ylabel(r"Flux density [$\mu$Jy]")
ax.set_ylim(-1, 10)
ax.set_title("RXCJ2248 source 2478_3 (z = 6.1052) — G395M")
plt.tight_layout()

## 2. Fit emission lines

Run a standard least-squares fit with broad Balmer component search.
This spectrum has strong [OIII] 4363 — the key auroral line for the
direct T_e method.

In [ ]:
result = jwspecfit.fit_lines(spec, z=6.1052, deg=3, n_boot=1000)

In [ ]:
print(f"Selected model: {result.selected_model}")
print(f"Lines fitted:   {len(result.lines)}")
print(f"chi2/dof:       {result.chi2:.2f}")
print()
print(f"{'Line':<18s} {'Flux':>12s} {'Err':>12s} {'SNR':>8s} {'EW (A)':>10s}")
print("-" * 65)
for name, lr in result.lines.items():
    tag = " *" if "BROAD" in name else ""
    print(
        f"{name + tag:<18s} {lr.flux:12.3e} {lr.flux_err:12.3e}"
        f" {lr.snr:8.1f} {lr.ew_A:10.1f}"
    )

In [ ]:
fig = jwspecfit.plot_fit(result, exclude_wave_A=[(20000, 28000), (50000, 60000)])
fig.suptitle("Line fit: RXCJ2248 2478_3 (G395M)", y=1.02)
plt.show()

## 3. Compute abundances — auto mode

The `compute_abundances` function automatically selects the direct T_e method
when [OIII] 4363 is detected above the SNR threshold (default 3). It also
derives A_V from the Balmer decrement before correcting all line fluxes.

In [ ]:
abund_auto = jwspecabund.compute_abundances(result, z=6.1052, n_mc=1000)
print(abund_auto.summary())

## 4. Direct method in detail

Explicitly run the direct method and inspect all intermediate quantities
(T_e, n_e, ionic abundances, ICFs).

In [ ]:
abund_direct = jwspecabund.compute_abundances(
    result, z=6.1052,
    method="direct",
    dust_law="salim",
    Te_relation="desi",
    n_mc=1000,
)

print(f"Method:        {abund_direct.method}")
print(f"A_V:           {abund_direct.Av:.3f}")
print(f"T_e(high):     {abund_direct.Te_high:.0f} K")
print(f"T_e(low):      {abund_direct.Te_low:.0f} K")
print(f"n_e:           {abund_direct.ne:.0f} cm^-3")
print(f"12+log(O/H):   {abund_direct.OH:.3f} +/- {abund_direct.OH_err:.3f}")
if abund_direct.NO is not None:
    print(f"log(N/O):      {abund_direct.NO:.3f} +/- {abund_direct.NO_err}")

In [ ]:
# Ionic abundances
if abund_direct.ionic:
    print("Ionic abundances:")
    for ion, val in abund_direct.ionic.items():
        print(f"  {ion:<12s} = {val:.3e}  (12+log = {12 + np.log10(val):.3f})")

In [ ]:
# MC posterior on 12+log(O/H)
if abund_direct.OH_posterior is not None:
    oh_post = abund_direct.OH_posterior
    oh_post = oh_post[np.isfinite(oh_post)]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(oh_post, bins=40, density=True, alpha=0.7, color="C0", edgecolor="C0")
    med = np.nanmedian(oh_post)
    lo = np.nanpercentile(oh_post, 16)
    hi = np.nanpercentile(oh_post, 84)
    ax.axvline(med, color="C1", ls="-", lw=1.5, label=f"Median = {med:.3f}")
    ax.axvline(lo, color="C1", ls="--", lw=1, label=f"16th = {lo:.3f}")
    ax.axvline(hi, color="C1", ls="--", lw=1, label=f"84th = {hi:.3f}")
    ax.set_xlabel("12 + log(O/H)")
    ax.set_ylabel("Probability density")
    ax.set_title("Direct T_e method: O/H posterior (MC)")
    ax.legend(fontsize=9)
    plt.tight_layout()

## 5. Strong-line method (Sanders+25)

For comparison, compute the metallicity using only the strong-line
calibrations — as would be done if [OIII] 4363 were undetected.

In [ ]:
abund_strong = jwspecabund.compute_abundances(
    result, z=6.1052,
    method="strong_line",
    dust_law="salim",
    n_mc=1000,
)

print(f"Method:        {abund_strong.method}")
print(f"A_V:           {abund_strong.Av:.3f}")
print(f"12+log(O/H):   {abund_strong.OH:.3f} (-{abund_strong.OH_err[0]:.3f}, +{abund_strong.OH_err[1]:.3f})")
print(f"Ratios used:   {abund_strong.ratios_used}")
print(f"chi2:          {abund_strong.chi2:.2f}")

## 6. Compare direct vs strong-line

Side-by-side comparison of the two methods.

In [ ]:
print(f"{'':>20s} {'Direct T_e':>18s} {'Strong-line':>18s}")
print("-" * 58)

# Format O/H errors
if isinstance(abund_direct.OH_err, tuple):
    d_err = f"(-{abund_direct.OH_err[0]:.3f}, +{abund_direct.OH_err[1]:.3f})"
else:
    d_err = f"+/- {abund_direct.OH_err:.3f}"

if isinstance(abund_strong.OH_err, tuple):
    s_err = f"(-{abund_strong.OH_err[0]:.3f}, +{abund_strong.OH_err[1]:.3f})"
else:
    s_err = f"+/- {abund_strong.OH_err:.3f}"

print(f"{'12+log(O/H)':>20s} {abund_direct.OH:>8.3f} {d_err:>9s} {abund_strong.OH:>8.3f} {s_err:>9s}")

if abund_direct.NO is not None:
    print(f"{'log(N/O)':>20s} {abund_direct.NO:>18.3f} {'---':>18s}")

if abund_direct.Te_high is not None:
    print(f"{'T_e(high) [K]':>20s} {abund_direct.Te_high:>18.0f} {'---':>18s}")
    print(f"{'T_e(low) [K]':>20s} {abund_direct.Te_low:>18.0f} {'---':>18s}")

print(f"{'A_V':>20s} {abund_direct.Av:>18.3f} {abund_strong.Av:>18.3f}")

In [ ]:
# Overlay O/H posteriors from both methods
fig, ax = plt.subplots(figsize=(7, 4))

if abund_direct.OH_posterior is not None:
    oh_d = abund_direct.OH_posterior[np.isfinite(abund_direct.OH_posterior)]
    ax.hist(oh_d, bins=40, density=True, alpha=0.5, color="C0",
            edgecolor="C0", label=f"Direct: {np.nanmedian(oh_d):.3f}")

if abund_strong.OH_posterior is not None:
    oh_s = abund_strong.OH_posterior[np.isfinite(abund_strong.OH_posterior)]
    ax.hist(oh_s, bins=40, density=True, alpha=0.5, color="C3",
            edgecolor="C3", label=f"Strong-line: {np.nanmedian(oh_s):.3f}")

ax.set_xlabel("12 + log(O/H)")
ax.set_ylabel("Probability density")
ax.set_title("Direct vs Strong-line metallicity")
ax.legend(fontsize=10)
plt.tight_layout()

## 7. Dust correction comparison

Compare Salim+18 and Cardelli+89 dust laws on the same data.

In [ ]:
abund_cardelli = jwspecabund.compute_abundances(
    result, z=6.1052,
    method="direct",
    dust_law="cardelli",
    n_mc=500,
)

print(f"{'Dust law':>15s} {'A_V':>8s} {'12+log(O/H)':>14s} {'T_e(high)':>12s}")
print("-" * 52)
print(f"{'Salim+18':>15s} {abund_direct.Av:>8.3f} {abund_direct.OH:>14.3f} {abund_direct.Te_high:>12.0f}")
print(f"{'Cardelli+89':>15s} {abund_cardelli.Av:>8.3f} {abund_cardelli.OH:>14.3f} {abund_cardelli.Te_high:>12.0f}")

## 8. Inspect dust correction curves

In [ ]:
wave_grid = np.linspace(1200, 10000, 500)
Av_test = 1.0

A_salim = jwspecabund.salim_attenuation(wave_grid, Av_test)
A_cardelli = jwspecabund.cardelli_extinction(wave_grid, Av_test)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(wave_grid / 1e4, A_salim, label="Salim+18 (A_V=1)", color="C0")
ax.plot(wave_grid / 1e4, A_cardelli, label="Cardelli+89 (A_V=1)", color="C3")

# Mark key emission lines
key_lines = {
    r"[OII]": 3728, r"H$\beta$": 4862, r"[OIII]4363": 4364,
    r"[OIII]5007": 5008, r"H$\alpha$": 6564, r"[SII]": 6727,
}
for label, wave in key_lines.items():
    ax.axvline(wave / 1e4, color="0.7", ls=":", lw=0.7)
    ax.text(wave / 1e4, ax.get_ylim()[1] * 0.95, label, fontsize=7,
            ha="center", va="top", rotation=90, color="0.4")

ax.set_xlabel(r"Rest wavelength [$\mu$m]")
ax.set_ylabel(r"A($\lambda$) [mag]")
ax.set_title("Dust attenuation/extinction curves")
ax.legend()
plt.tight_layout()

## 9. No dust correction

For reference, compute the uncorrected abundance to see the effect of dust.

In [ ]:
abund_nodust = jwspecabund.compute_abundances(
    result, z=6.1052,
    method="direct",
    dust_correct=False,
    n_mc=500,
)

print(f"{'':>18s} {'With dust corr':>16s} {'No dust corr':>16s}")
print("-" * 52)
print(f"{'12+log(O/H)':>18s} {abund_direct.OH:>16.3f} {abund_nodust.OH:>16.3f}")
if abund_direct.Te_high is not None and abund_nodust.Te_high is not None:
    print(f"{'T_e(high) [K]':>18s} {abund_direct.Te_high:>16.0f} {abund_nodust.Te_high:>16.0f}")
if abund_direct.NO is not None and abund_nodust.NO is not None:
    print(f"{'log(N/O)':>18s} {abund_direct.NO:>16.3f} {abund_nodust.NO:>16.3f}")

## 10. Key line ratios for cross-checking

Print the raw and dust-corrected line ratios that feed into the
abundance calculation. Compare these with published values.

In [ ]:
lines = result.lines

def ratio_and_err(name_a, name_b):
    """Compute flux ratio and error from the fit result."""
    if name_a not in lines or name_b not in lines:
        return np.nan, np.nan
    fa = lines[name_a].flux
    fb = lines[name_b].flux
    ea = lines[name_a].flux_err
    eb = lines[name_b].flux_err
    if fb <= 0:
        return np.nan, np.nan
    r = fa / fb
    r_err = r * np.sqrt((ea / fa) ** 2 + (eb / fb) ** 2)
    return r, r_err

print("Observed (uncorrected) line ratios:")
print(f"{'Ratio':<30s} {'Value':>10s} {'Error':>10s}")
print("-" * 52)

ratio_list = [
    ("Ha / Hb",               "Ha",         "HBETA"),
    ("[OIII]5007 / Hb",       "OIII_5007",  "HBETA"),
    ("[OIII]4363 / [OIII]5007","OIII_4363", "OIII_5007"),
    ("[NII]6585 / Ha",        "NII_6585",   "Ha"),
    ("[SII]6718 / [SII]6732", "SII_6718",   "SII_6732"),
    ("HeI 5877 / Hb",         "HEI_5877",   "HBETA"),
]

for label, a, b in ratio_list:
    r, e = ratio_and_err(a, b)
    if np.isfinite(r):
        print(f"{label:<30s} {r:>10.4f} {e:>10.4f}")
    else:
        print(f"{label:<30s} {'---':>10s} {'---':>10s}")

## 11. Strong-line diagnostics breakdown

Show the individual diagnostic ratios and how they compare to the
Sanders+25 calibration curves.

In [ ]:
from jwspecabund.strong_line import CALIBRATIONS, Z_REF, compute_line_ratios

# Extract fluxes from the fit result
fluxes = {name: lr.flux for name, lr in result.lines.items() if "BROAD" not in name}
errors = {name: lr.flux_err for name, lr in result.lines.items() if "BROAD" not in name}

ratios = compute_line_ratios(fluxes, errors)

print(f"Available diagnostics: {list(ratios.keys())}")
print()
for name, data in ratios.items():
    print(f"  {name}: log(R) = {data['val']:.3f} +/- {data['err']:.3f}")

In [ ]:
# Plot Sanders+25 calibration curves vs the measured ratios
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
z_grid = np.linspace(6.5, 9.0, 200)

for ax, (diag_name, cal) in zip(axes.ravel(), CALIBRATIONS.items()):
    # Calibration curve
    y_model = [sum(c * (z - Z_REF) ** i for i, c in enumerate(cal["coeffs"])) for z in z_grid]
    ax.plot(z_grid, y_model, "k-", lw=1.5, label=f"Sanders+25 {diag_name}")
    ax.fill_between(
        z_grid,
        np.array(y_model) - cal["sigma_cal"],
        np.array(y_model) + cal["sigma_cal"],
        alpha=0.15, color="0.5",
    )

    # Measured ratio
    if diag_name in ratios:
        ax.axhline(ratios[diag_name]["val"], color="C0", ls="-", lw=1)
        ax.axhspan(
            ratios[diag_name]["val"] - ratios[diag_name]["err"],
            ratios[diag_name]["val"] + ratios[diag_name]["err"],
            alpha=0.2, color="C0",
        )

    # Best-fit metallicity
    ax.axvline(abund_strong.OH, color="C1", ls="--", lw=1, label=f"Z = {abund_strong.OH:.2f}")

    ax.set_xlabel("12 + log(O/H)")
    ax.set_ylabel(f"log({diag_name})")
    ax.legend(fontsize=8)

plt.suptitle("Sanders+25 calibration curves vs measured ratios", y=1.01)
plt.tight_layout()

## 12. Summary table for paper comparison

Compile all key measurements into a single summary table.

In [ ]:
print("=" * 60)
print("ABUNDANCE SUMMARY: RXCJ2248 source 2478_3 (z = 6.1052)")
print("=" * 60)
print()
print("--- Direct T_e method (Salim+18 dust, DESI T_e-T_e) ---")
print(f"  A_V            = {abund_direct.Av:.3f}")
print(f"  T_e(O++)       = {abund_direct.Te_high:.0f} K")
print(f"  T_e(O+)        = {abund_direct.Te_low:.0f} K")
print(f"  n_e            = {abund_direct.ne:.0f} cm^-3")
print(f"  12+log(O/H)    = {abund_direct.OH:.3f} +/- {abund_direct.OH_err}")
if abund_direct.NO is not None:
    print(f"  log(N/O)       = {abund_direct.NO:.3f} +/- {abund_direct.NO_err}")
if abund_direct.ionic:
    for ion, val in abund_direct.ionic.items():
        print(f"  {ion:<14s}  = {val:.3e}")

print()
print("--- Strong-line method (Sanders+25) ---")
print(f"  12+log(O/H)    = {abund_strong.OH:.3f} (-{abund_strong.OH_err[0]:.3f}, +{abund_strong.OH_err[1]:.3f})")
print(f"  Ratios used    = {abund_strong.ratios_used}")
print(f"  chi2           = {abund_strong.chi2:.2f}")
print()
print("--- Comparison ---")
diff = abund_direct.OH - abund_strong.OH
print(f"  Delta(O/H)     = {diff:+.3f} dex (direct - strong-line)")